# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

# 1. Setup (~1 minute)

In [ ]:
# ============================================================
# Real-ESRGAN en Google Colab (Python 3.12 / 3.13, torch 2.x)
# Pega TODO esto en UNA celda de Colab y ejecútala.
# Es seguro volver a ejecutarla (los parches son idempotentes).
# ============================================================
import os, sys, importlib
import torch
import cv2
import subprocess
from tqdm import tqdm
from PIL import Image

assert torch.cuda.is_available(), "GPU no detectada. Ve a Entorno de ejecución -> Cambiar tipo de entorno de ejecución y selecciona GPU."
print("Python:", sys.version.split()[0], "| torch:", torch.__version__)

# NO se actualiza setuptools (torch exige setuptools<82 y el upgrade rompía cosas).

def clone_and_install(url, path):
    """Clona el repo, parchea su setup.py y lo instala sin tocar torch/numpy."""
    if not os.path.exists(path):
        !git clone -q {url} {path}
    # (1) setup_requires intenta compilar torch/numpy/cython en un entorno aislado -> falla
    !sed -i "/setup_requires=/d" {path}/setup.py
    # (2) Python 3.13: exec() ya no crea variables locales -> KeyError '__version__'
    !sed -i "s/return locals()\['__version__'\]/return open('VERSION').read().strip()/" {path}/setup.py
    # (3) tb-nightly ya no se necesita (TensorBoard)
    !sed -i 's/tb-nightly//g' {path}/requirements.txt
    # (4) --no-deps y --no-build-isolation: no reinstalar torch/numpy ni bajar otro basicsr de PyPI
    !cd {path} && pip install -q --no-deps --no-build-isolation -e .

# Dependencias que antes traían basicsr / gfpgan / facexlib (se instalan a mano)
print("Instalando dependencias...")
!pip install -q addict future lmdb yapf pyyaml requests tqdm filterpy numba scikit-image scipy ffmpeg-python

print("Instalando BasicSR, facexlib, GFPGAN y Real-ESRGAN...")
clone_and_install("https://github.com/xinntao/BasicSR.git",      "/content/BasicSR")
clone_and_install("https://github.com/xinntao/facexlib.git",     "/content/facexlib")
clone_and_install("https://github.com/TencentARC/GFPGAN.git",    "/content/GFPGAN")
clone_and_install("https://github.com/xinntao/Real-ESRGAN.git",  "/content/Real-ESRGAN")

# Los instalados con -e no se ven en el kernel ya iniciado hasta reiniciarlo:
# se añaden las rutas a mano para poder importarlos YA, sin reiniciar.
for p in ["/content/BasicSR", "/content/facexlib", "/content/GFPGAN", "/content/Real-ESRGAN"]:
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

%cd /content/Real-ESRGAN

# Parches de compatibilidad (inofensivos si ya no hacen falta)
import basicsr
degr = os.path.join(basicsr.__path__[0], 'data', 'degradations.py')
!sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/' {degr}
# numpy >= 1.24 eliminó np.float (lo usa facexlib en un archivo de alineación)
!sed -i 's/astype(np.float,/astype(float,/' /content/facexlib/facexlib/alignment/awing_arch.py

# Descargar modelo por defecto
model_dir = 'experiments/pretrained_models'
os.makedirs(model_dir, exist_ok=True)
if not os.path.exists(os.path.join(model_dir, 'RealESRGAN_x4plus.pth')):
    print("Descargando modelo RealESRGAN_x4plus...")
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P {model_dir}

# Verificación final: si esto imprime OK, todo quedó bien instalado
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
import facexlib, gfpgan
print("Imports OK -> basicsr, realesrgan, facexlib, gfpgan")

mount_drive = False
print("\n¡Configuración completada con éxito y optimizada para Colab!")


# 2. Mount drive (optional)

In [ ]:
from google.colab import drive
mount_drive=False #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [ ]:
video_path="/content/drive/MyDrive/mis_videos" #@param{type:"string"}
output_dir="/content/" #@param{type:"string"}
resolution = "FHD (1920 x 1080)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"] {type:"string"}
model = "realesr-animevideov3" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3", "realesr-general-x4v3"]

# video_path puede ser un VIDEO suelto o una CARPETA con videos (los procesa uno por uno).
# Si ya existe el resultado de un video, lo salta: puedes volver a ejecutar la celda tras una desconexión.
# Usa una ruta de Drive en output_dir: /content se borra al cerrar la sesión.

VIDEO_EXTS = (".mp4", ".mov", ".mkv", ".webm", ".avi")
REPO_DIR = "/content/Real-ESRGAN"

if not output_dir.endswith("/"):
  output_dir += "/"
os.makedirs(output_dir, exist_ok=True)


def upscale_video(video_path):
  assert os.path.exists(video_path), "Video file does not exist"

  video_capture = cv2.VideoCapture(video_path)
  video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
  video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
  video_capture.release()

  final_width = None
  final_height = None
  aspect_ratio = float(video_width/video_height)

  # Get output resolutions
  match resolution:
    case "FHD (1920 x 1080)":
      final_width=1920
      final_height=1080
    case "2k (2560 x 1440)":
      final_width=2560
      final_height=1440
    case "4k (3840 x 2160)":
      final_width=3840
      final_height=2160
    case "2 x original":
      final_width=2*video_width
      final_height=2*video_height
    case "3 x original":
      final_width=3*video_width
      final_height=3*video_height
    case "4 x original":
      final_width=4*video_width
      final_height=4*video_height

  if aspect_ratio == 1.0 and "original" not in resolution:
    final_height = final_width

  if aspect_ratio < 1.0 and "original" not in resolution:
    temp = final_width
    final_width = final_height
    final_height = temp

  video_name = os.path.splitext(os.path.basename(video_path))[0]
  final_video_name = f"{video_name}_upscaled_{final_width}_{final_height}.mp4"
  final_video_path = os.path.join(output_dir, final_video_name)

  if os.path.exists(final_video_path):
    print("   Ya existe el resultado, salto.")
    return

  scale_factor = max(final_width/video_width, final_height/video_height)
  isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

  # scale_factor needs to be even
  while isEven == False:
    scale_factor += 0.01
    isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

  print(f"   Upscaling from {video_width}x{video_height} to {final_width}x{final_height}, scale_factor={scale_factor}")

  subprocess.run(
      ["python", "inference_realesrgan_video.py", "-n", model,
       "-i", video_path, "-o", output_dir, "--outscale", str(scale_factor)],
      cwd=REPO_DIR)

  upscaled_video_path = f"{output_dir}{video_name}_out.mp4"
  assert os.path.exists(upscaled_video_path), "La inferencia no generó el video"

  # crop to fit
  if "original" not in resolution:
    print("   Cropping to fit...")
    command = f"ffmpeg -loglevel error -hwaccel cuda -y -i '{upscaled_video_path}' -c:v h264_nvenc -filter:v  'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2' -c:v libx264 -pix_fmt yuv420p '{final_video_path}'"
    subprocess.run(command,shell=True)
  else:
    # final video path = upscaled video path
    command = f"cp '{upscaled_video_path}' '{final_video_path}'"
    subprocess.run(command,shell=True)

  print(f"   Upscaled video saved to: {final_video_path}")

  # save to drive
  if mount_drive:
    drive_folder = "MyDrive/Upscaled Videos (REAL-ESRGAN)"
    save_directory_drive = f"/content/drive/{drive_folder}"   # tu Drive está montado en /content/drive
    os.makedirs(save_directory_drive, exist_ok=True)

    command = f"cp '{final_video_path}' '{save_directory_drive}/{final_video_name}'"
    subprocess.run(command,shell=True)
    print(f"   Saved to drive: /{drive_folder}/{final_video_name}" )

  os.remove(upscaled_video_path)


# ---- Recorrer carpeta (o video suelto) ----
if os.path.isdir(video_path):
  videos = sorted(
      os.path.join(video_path, f) for f in os.listdir(video_path)
      if f.lower().endswith(VIDEO_EXTS) and "_upscaled_" not in f)
else:
  videos = [video_path]

assert len(videos) > 0, "No se encontraron videos"
print(f"{len(videos)} video(s) a procesar con {model}")

for i, v in enumerate(videos, 1):
  print(f"\n[{i}/{len(videos)}] {os.path.basename(v)}")
  try:
    upscale_video(v)
  except Exception as e:
    print(f"   ERROR en {os.path.basename(v)}: {e}. Continúo con el siguiente.")

print("\nListo.")

In [ ]:
import shutil
from google.colab import files

# 1. Define la ruta de tu carpeta y el nombre del archivo zip resultante
carpeta_a_comprimir = '/content/'  # Cambia esto por tu ruta
nombre_zip = 'archivo'

# 2. Comprime la carpeta en un archivo .zip
shutil.make_archive(nombre_zip, 'zip', carpeta_a_comprimir)

# 3. Descarga el archivo zip generado
files.download(f'{nombre_zip}.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 4. Disconnect runtime

In [ ]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()